In [5]:
!pip install -q streamlit

In [7]:
import os
import pandas as pd

# Check both /ML/... and /content/ML/...
candidate_paths = "/content/sample_data/SriLanka_Weather_Dataset_V1.csv",

dataset_path = next((p for p in candidate_paths if os.path.exists(p)), None)

if dataset_path is None:
  raise FileNotFoundError(
      "Dataset not found! Check left Files pane or run: !find / -name"
      " 'SriLanka_Weather_Dataset_V1.csv' 2>/dev/null"
  )

print("Using dataset:", dataset_path)
df = pd.read_csv(dataset_path)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head(3)

Using dataset: /content/sample_data/SriLanka_Weather_Dataset_V1.csv
Shape: (147480, 24)
Columns: ['time', 'weathercode', 'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean', 'apparent_temperature_max', 'apparent_temperature_min', 'apparent_temperature_mean', 'sunrise', 'sunset', 'shortwave_radiation_sum', 'precipitation_sum', 'rain_sum', 'snowfall_sum', 'precipitation_hours', 'windspeed_10m_max', 'windgusts_10m_max', 'winddirection_10m_dominant', 'et0_fao_evapotranspiration', 'latitude', 'longitude', 'elevation', 'country', 'city']


,time,weathercode,temperature_2m_max,temperature_2m_min,temperature_2m_mean,apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,sunrise,sunset,...,precipitation_hours,windspeed_10m_max,windgusts_10m_max,winddirection_10m_dominant,et0_fao_evapotranspiration,latitude,longitude,elevation,country,city
0,1/1/2010,2,30.0,22.7,26.1,34.4,25.2,29.2,6:22:00 AM,6:05:00 PM,...,0,11.7,27.4,20,4.58,7.0,79.899994,16,Sri Lanka,Colombo
1,1/2/2010,51,29.9,23.5,26.2,33.8,26.2,29.8,6:22:00 AM,6:06:00 PM,...,1,13.0,27.0,24,3.84,7.0,79.899994,16,Sri Lanka,Colombo
2,1/3/2010,51,29.5,23.2,26.0,34.3,26.3,29.9,6:23:00 AM,6:06:00 PM,...,3,12.3,27.4,16,3.65,7.0,79.899994,16,Sri Lanka,Colombo


In [8]:
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    BaggingRegressor,
    GradientBoostingRegressor,
    RandomForestRegressor,
    StackingRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

# Date & seasonal features
df["time"] = pd.to_datetime(df["time"])
df["month"] = df["time"].dt.month
df["day_of_year"] = df["time"].dt.dayofyear

target = "precipitation_sum"
num_features = [
    "temperature_2m_max",
    "temperature_2m_min",
    "shortwave_radiation_sum",
    "windspeed_10m_max",
    "et0_fao_evapotranspiration",
    "elevation",
    "month",
    "day_of_year",
]
cat_features = ["city"]

# Filter to columns present in your CSV (safe check)
num_features = [c for c in num_features if c in df.columns]
cat_features = [c for c in cat_features if c in df.columns]

df_clean = df[num_features + cat_features + [target]].dropna(subset=[target])
df_sample = df_clean.sample(n=min(25000, len(df_clean)), random_state=42)

X = df_sample[num_features + cat_features]
y = df_sample[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            num_features,
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            cat_features,
        ),
    ]
)

bagging_model = Pipeline([
    ("prep", preprocessor),
    (
        "model",
        BaggingRegressor(
            estimator=DecisionTreeRegressor(max_depth=12, random_state=42),
            n_estimators=50,
            random_state=42,
            n_jobs=-1,
        ),
    ),
])

boosting_model = Pipeline([
    ("prep", preprocessor),
    (
        "model",
        GradientBoostingRegressor(
            n_estimators=150,
            learning_rate=0.08,
            max_depth=5,
            random_state=42,
        ),
    ),
])

stacking_model = Pipeline([
    ("prep", preprocessor),
    (
        "model",
        StackingRegressor(
            estimators=[
                (
                    "rf",
                    RandomForestRegressor(
                        n_estimators=60, max_depth=12, random_state=42, n_jobs=-1
                    ),
                ),
                (
                    "gbr",
                    GradientBoostingRegressor(
                        n_estimators=100, max_depth=4, random_state=42
                    ),
                ),
                ("ridge", Ridge(alpha=1.0)),
            ],
            final_estimator=Ridge(alpha=1.0),
            cv=3,
            n_jobs=-1,
        ),
    ),
])

models = {
    "Bagging (Decision Trees)": bagging_model,
    "Boosting (Gradient Boosting)": boosting_model,
    "Stacking (RF + GBR + Ridge)": stacking_model,
}

results = []
trained_models = {}

for name, pipe in models.items():
  pipe.fit(X_train, y_train)
  preds = pipe.predict(X_test)
  rmse = np.sqrt(mean_squared_error(y_test, preds))
  mae = mean_absolute_error(y_test, preds)
  r2 = r2_score(y_test, preds)

  results.append({
      "Model": name,
      "RMSE (mm)": round(rmse, 3),
      "MAE (mm)": round(mae, 3),
      "R2 Score": round(r2, 4),
  })
  trained_models[name] = pipe
  print(f"{name} -> RMSE: {rmse:.3f} | MAE: {mae:.3f} | R2: {r2:.4f}")

metrics_df = pd.DataFrame(results).sort_values("RMSE (mm)")
metrics_df.to_csv("/content/metrics.csv", index=False)
joblib.dump(trained_models, "/content/trained_models.joblib")

cities_list = (
    sorted(df_sample["city"].dropna().unique().tolist())
    if "city" in df_sample.columns
    else ["Colombo"]
)
joblib.dump(cities_list, "/content/cities.joblib")

print("Saved /content/metrics.csv, /content/trained_models.joblib, and /content/cities.joblib")

Bagging (Decision Trees) -> RMSE: 6.693 | MAE: 3.234 | R2: 0.5061
Boosting (Gradient Boosting) -> RMSE: 6.590 | MAE: 3.238 | R2: 0.5212
Stacking (RF + GBR + Ridge) -> RMSE: 6.648 | MAE: 3.230 | R2: 0.5127
Saved /content/metrics.csv, /content/trained_models.joblib, and /content/cities.joblib


In [9]:
%%writefile /content/app.py
import os
import joblib
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Sri Lanka Rainfall Ensemble Dashboard", layout="wide")

st.title("🇱🇰 Sri Lanka Rainfall Prediction — Ensemble Learning Dashboard")
st.markdown(
    "Comparing **Bagging**, **Boosting**, and **Stacking** models trained on meteorological data across Sri Lankan cities."
)

def resolve_file(filename):
    for path in [filename, os.path.join("/content", filename)]:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"Missing required artifact: {filename}")

metrics_df = pd.read_csv(resolve_file("metrics.csv"))
models = joblib.load(resolve_file("trained_models.joblib"))
cities = joblib.load(resolve_file("cities.joblib"))

st.subheader("1. Ensemble Model Comparison")
col1, col2 = st.columns([1.2, 1])

with col1:
    st.dataframe(metrics_df, use_container_width=True, hide_index=True)

with col2:
    st.bar_chart(metrics_df.set_index("Model")["R2 Score"])

best_model_name = metrics_df.iloc[0]["Model"]
st.info(f"🏆 **Top Performing Ensemble (Lowest RMSE):** {best_model_name}")

st.divider()

st.subheader("2. Interactive Weather & Rainfall Simulator")

c1, c2, c3 = st.columns(3)
with c1:
    city = st.selectbox(
        "Sri Lankan City",
        cities,
        index=cities.index("Colombo") if "Colombo" in cities else 0,
    )
    month = st.slider("Month (1 = Jan, 12 = Dec)", 1, 12, 5)
    elevation = st.number_input("City Elevation (m)", min_value=0, max_value=2500, value=10)

with c2:
    temp_max = st.slider("Max Temperature (°C)", 15.0, 40.0, 31.0)
    temp_min = st.slider("Min Temperature (°C)", 10.0, 32.0, 25.0)
    wind = st.slider("Max Wind Speed (km/h)", 0.0, 60.0, 18.0)

with c3:
    radiation = st.slider("Shortwave Radiation (MJ/m²)", 5.0, 30.0, 17.5)
    et0 = st.slider("Evapotranspiration ET0 (mm)", 1.0, 8.0, 3.8)
    selected_model = st.selectbox("Select Ensemble Model", list(models.keys()))

input_df = pd.DataFrame([{
    "temperature_2m_max": temp_max,
    "temperature_2m_min": temp_min,
    "shortwave_radiation_sum": radiation,
    "windspeed_10m_max": wind,
    "et0_fao_evapotranspiration": et0,
    "elevation": elevation,
    "month": month,
    "day_of_year": month * 30 - 15,
    "city": city,
}])

if st.button("Predict Daily Rainfall", type="primary"):
    pred = max(0.0, float(models[selected_model].predict(input_df)[0]))
    st.success(f"**Predicted Rainfall in {city} ({selected_model}):** `{pred:.2f} mm`")

    all_preds = {
        m_name: round(max(0.0, float(m.predict(input_df)[0])), 2)
        for m_name, m in models.items()
    }
    st.write("**Comparison across all three ensemble methods (mm):**", all_preds)

Writing /content/app.py


In [10]:
%%writefile /content/requirements.txt
streamlit
scikit-learn
pandas
numpy
joblib

Writing /content/requirements.txt
